# By using dataloader
* Glass Dataset

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import os
os.chdir("D:\Machine Learning\Datasets\Cases\Glass_Identification")
import torch 
import torch.nn as nn
from torchsummary import summary
from torch.utils.data import DataLoader, TensorDataset

#  sklearn imports 
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss, r2_score
glass = pd.read_csv('Glass.csv')
dum_gls = pd.get_dummies(glass)
X = dum_gls.iloc[:,:-6]
y = dum_gls.iloc[:,-6:]
scaler = MinMaxScaler()
X_train, X_test, y_train, y_test = train_test_split(X,y.values, random_state =25, test_size = 0.3, stratify=glass['Type'])
X_scl_trn = scaler.fit_transform(X_train)
X_scl_tst = scaler.transform(X_test)
X_scl_trn.shape
X_torch = torch.from_numpy(X_scl_trn)
y_torch = torch.from_numpy(y_train)
print(X_torch.size())
print(y_torch.size())
torch.manual_seed(25)
##Data Loader 
# Data loader code
joint_dataset = TensorDataset(X_torch.float(), y_torch.float())
torch.manual_seed(25)
data_loader = DataLoader(dataset=joint_dataset, batch_size=149, shuffle=True)
## Model Defination

model = nn.Sequential(nn.Linear(in_features=X_scl_trn.shape[1], out_features=18),
                    
                    nn.ReLU()
                    ,nn.Linear(in_features=18, out_features=6)
                    )
summary(model, (1,X_scl_trn.shape[1]))                    
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.05)
optimizer
#### Generating Predictions with default weights
y_pred = model(X_torch.float()) # Weighted Sums aka lin_outputs not actual probas
y_pred[:3]
print(y_torch.shape)
loss = criterion(y_pred, y_torch.float())
loss
# for epoch in np.arange(0,1000):
#     # Forward Pass:  Compute predicted y by passing x to the model
#     y_pred_prob = model(X_torch.float()) # Inferencing of X
#     # Compute and print loss
#     loss = criterion(y_pred_prob, y_torch.float())
#     if epoch % 100 == 0:
#         print('Epoch', epoch+1, 'Loss:', loss.item())
#     # Zero Gradients, perform a backward pass, and update the weights.
#     optimizer.zero_grad() # all gradients are equaled to 0
#     #perform a backward pass (backpropogation)
#     loss.backward()
#     #
#     optimizer.step()
for epoch in np.arange(0,1000):
    for i, batch in enumerate(data_loader,1):
        y_pred_prob = model(X_torch.float()) # Inferencing of X
        loss = criterion(y_pred_prob, batch[1].float().unsqueeze(1))
        if epoch % 100 == 0:
            print('Epoch', epoch+1, 'Loss:', loss.item())
        optimizer.zero_grad() # all gradients are equaled to 0
        loss.backward()
        optimizer.step()

X_torch_test = torch.from_numpy(X_scl_tst)
type(X_torch_test)
lin_output = model(X_torch_test.float()) # Equivalent predict_proba
type(lin_output)

y_wt_sum = model(X_torch_test.float())
softmax = nn.Softmax(dim=1)
pred_proba = softmax(y_wt_sum)
pred_proba
y_pred_scl = pred_proba.detach().numpy()
y_pred_scl[:3]
y_pred = np.argmax(pred_proba.detach().numpy(), axis = 1)
y_pred
y_test_org = np.argmax(y_test, axis = 1)
print(accuracy_score(y_test_org,y_pred))
log_loss(y_test_org, y_pred_prob)

torch.Size([149, 9])
torch.Size([149, 6])
----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                [-1, 1, 18]             180
              ReLU-2                [-1, 1, 18]               0
            Linear-3                 [-1, 1, 6]             114
Total params: 294
Trainable params: 294
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------
torch.Size([149, 6])


RuntimeError: 0D or 1D target tensor expected, multi-target not supported

In [5]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
import os
os.chdir(r"D:\Machine Learning\Datasets\Cases\Glass_Identification")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchsummary import summary

# sklearn imports
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

# Load data
glass = pd.read_csv('Glass.csv')

# Features: all columns except 'Type'
X = glass.drop('Type', axis=1)

# Target: encode Type as integer labels (0 to 5)
le = LabelEncoder()
y = le.fit_transform(glass['Type'])

# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=25, stratify=y
)

# Scale features
scaler = MinMaxScaler()
X_scl_trn = scaler.fit_transform(X_train)
X_scl_tst = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_tensor = torch.from_numpy(X_scl_trn).float()
y_train_tensor = torch.from_numpy(y_train).long()        # IMPORTANT: long for class indices
X_test_tensor = torch.from_numpy(X_scl_tst).float()
y_test_tensor = torch.from_numpy(y_test).long()

# Dataset and DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)  # Smaller batch size

# Model definition
torch.manual_seed(25)
model = nn.Sequential(
    nn.Linear(in_features=X_scl_trn.shape[1], out_features=18),
    nn.ReLU(),
    nn.Linear(in_features=18, out_features=6)  # 6 classes
)

# Print model summary
summary(model, input_size=(X_scl_trn.shape[1],))  # Correct shape

# Loss and optimizer
criterion = nn.CrossEntropyLoss()          # Expects raw logits and long labels
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

# Training loop
num_epochs = 1000
for epoch in range(num_epochs):
    model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        
        # Forward pass
        outputs = model(batch_X)           # Raw logits
        
        # Compute loss
        loss = criterion(outputs, batch_y)
        
        # Backward pass
        loss.backward()
        optimizer.step()
    
    # Print loss every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

# Evaluation on test set
model.eval()
with torch.no_grad():
    test_outputs = model(X_test_tensor)
    
    # Probabilities
    softmax = nn.Softmax(dim=1)
    test_proba = softmax(test_outputs)
    
    # Predicted class
    y_pred = torch.argmax(test_outputs, dim=1).numpy()  # Directly from logits is fine
    
    # Accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f'\nTest Accuracy: {accuracy:.4f}')
    
    # Log loss (using probabilities)
    test_proba_np = test_proba.numpy()
    test_log_loss = log_loss(y_test, test_proba_np)
    print(f'Test Log Loss: {test_log_loss:.4f}')

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                   [-1, 18]             180
              ReLU-2                   [-1, 18]               0
            Linear-3                    [-1, 6]             114
Total params: 294
Trainable params: 294
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------
Epoch [100/1000], Loss: 1.2826
Epoch [200/1000], Loss: 0.7474
Epoch [300/1000], Loss: 1.0966
Epoch [400/1000], Loss: 1.1910
Epoch [500/1000], Loss: 0.8343
Epoch [600/1000], Loss: 0.7013
Epoch [700/1000], Loss: 0.7881
Epoch [800/1000], Loss: 0.7395
Epoch [900/1000], Loss: 0.8602
Epoch [1000/1000], Loss: 0.8553

Test Accuracy: 0.6615
Test Log Loss: 0.9306
